# Vendor Ranking and Cost Efficiency Deep Dive

This notebook turns the curated summary into a decision model for vendor prioritization. It is designed to answer a practical management question: which vendors should be scaled, renegotiated, or reviewed first?


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="ticks", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 6)


In [ ]:
vendor_summary = pd.read_csv("../data/processed/vendor_performance_summary.csv")
vendor_brand = pd.read_csv("../data/processed/vendor_brand_performance.csv")

vendor_summary.head()


## Advanced Analysis: Composite Vendor Ranking

The performance score blends profit, margin, stock turnover, revenue contribution, and lead-time discipline. This is useful because a single metric usually over-rewards either scale or efficiency, but not both.


In [ ]:
score_columns = [
    "PerformanceScore",
    "AdjustedProfit",
    "AdjustedMarginPct",
    "StockTurnover",
    "AvgLeadDays",
    "Revenue",
]
vendor_summary[score_columns].describe().T


In [ ]:
top_ranked = vendor_summary.nsmallest(15, "PerformanceRank").copy()
top_ranked[["VendorName", "PerformanceRank", "PerformanceScore", "AdjustedProfit", "AdjustedMarginPct", "StockTurnover"]]


In [ ]:
plt.figure(figsize=(12, 7))
ranked_view = vendor_summary.nsmallest(15, "PerformanceRank").sort_values("PerformanceScore")
sns.barplot(data=ranked_view, x="PerformanceScore", y="VendorName", hue="PerformanceTier", dodge=False)
plt.title("Top 15 Vendors by Composite Performance Score")
plt.xlabel("Performance Score")
plt.ylabel("Vendor")
plt.legend(title="Tier", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()


## Cost Efficiency Matrix

This view highlights suppliers that absorb meaningful spend but fail to convert that spend into proportionate margin. Those are usually the right candidates for pricing review, contract negotiation, or SKU rationalization.


In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=vendor_summary,
    x="CostSharePct",
    y="AdjustedMarginPct",
    size="Revenue",
    hue="PerformanceTier",
    sizes=(50, 600),
    alpha=0.8,
)
plt.axvline(vendor_summary["CostSharePct"].median(), color="grey", linestyle="--", linewidth=1)
plt.axhline(vendor_summary["AdjustedMarginPct"].median(), color="grey", linestyle="--", linewidth=1)
plt.title("Cost Share vs Adjusted Margin")
plt.xlabel("Share of Total Procurement Cost (%)")
plt.ylabel("Adjusted Margin (%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()


## Purchase-to-Sales Mismatch Review

This section isolates vendors where procurement has moved faster than sell-through. That usually signals excess stock, weaker assortment discipline, or demand planning issues.


In [ ]:
mismatch = vendor_summary.sort_values(["InventoryAtRisk", "UnsoldQuantity"], ascending=False).head(15)
mismatch[["VendorName", "PurchaseQuantity", "SalesQuantity", "UnsoldQuantity", "InventoryAtRisk", "SalesToPurchaseRatio"]]


In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(
    data=mismatch.sort_values("InventoryAtRisk"),
    x="InventoryAtRisk",
    y="VendorName",
    color="#A23B72",
)
plt.title("Top 15 Vendors by Inventory at Risk")
plt.xlabel("Inventory at Risk")
plt.ylabel("Vendor")
plt.show()


## Underperforming Vendor Drill-Down

Looking one level deeper helps explain *why* a vendor is underperforming. Often the issue is not the whole relationship, but a small cluster of brands dragging down the economics.


In [ ]:
weak_vendors = vendor_summary.nsmallest(5, "AdjustedProfit")["VendorNumber"]
weak_brand_mix = (
    vendor_brand[vendor_brand["VendorNumber"].isin(weak_vendors)]
    .sort_values(["VendorNumber", "AdjustedProfit"])
    [["VendorName", "Brand", "Classification", "Revenue", "PurchaseCost", "AdjustedProfit", "InventoryAtRisk", "AvgLeadDays"]]
)
weak_brand_mix.head(20)


## Decision Framing

This ranking framework supports three actions:

1. Protect top-tier vendors that combine scale, margin, and operational reliability.
2. Renegotiate cost-heavy vendors with weak adjusted margins.
3. Reduce exposure to vendors that repeatedly create excess stock and slow replenishment cycles.
